# GX_00_setup

Create schema and volume

In [2]:
   %sql
   CREATE SCHEMA IF NOT EXISTS workspace.churn;
   CREATE VOLUME IF NOT EXISTS workspace.churn.raw;

SyntaxError: invalid syntax (1765658785.py, line 2)

In [1]:
import os
import sys
import tempfile
from pathlib import Path

# Usar o Java e o Python do ambiente Conda deste notebook.
os.environ["JAVA_HOME"] = sys.prefix
os.environ["PATH"] = str(Path(sys.prefix) / "bin") + os.pathsep + os.environ["PATH"]
os.environ["PYSPARK_PYTHON"] = sys.executable

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("GraphFramesLocal")
    .config(
        "spark.jars.packages",
        "io.graphframes:graphframes-spark3_2.12:0.12.3"
    )
    .getOrCreate()
)

spark.sparkContext.setCheckpointDir(
    tempfile.mkdtemp(prefix="graphframes-checkpoints-")
)

print("Spark:", spark.version)
print("Python:", sys.version.split()[0])

/Users/joanaanmartins/opt/anaconda3/envs/graphframes-local/lib/python3.11/site-packages/pyspark/bin/spark-class: line 71: /Users/joanaanmartins/opt/anaconda3/envs/graphframes-local/bin/java: No such file or directory
/Users/joanaanmartins/opt/anaconda3/envs/graphframes-local/lib/python3.11/site-packages/pyspark/bin/spark-class: line 97: CMD: bad array subscript
head: illegal line count -- -1


PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

Version check and row counts

In [ ]:
import sys

RAW = "/Volumes/workspace/churn/raw"

print("Spark:", spark.version)
print("Python:", sys.version.split()[0])

churn = (spark.read
        .option("header", True)
        .option("encoding", "ISO-8859-1")
        .csv(f"{RAW}/churn.csv"))

tx = (spark.read
        .option("header", True)
        .option("quote", '"')
        .option("escape", '"')
        .csv(f"{RAW}/bank_transactions.csv"))

n_churn, n_tx = churn.count(), tx.count()
print("churn rows:", n_churn, "| cols:", len(churn.columns))
print("tx rows:   ", n_tx, "| cols:", len(tx.columns))

assert n_churn == 10_000, "churn row count wrong"
assert n_tx == 1_000, "transactions row count wrong"
print("Smoke test passed")

Install GraphFrames

In [2]:
%pip install graphframes-py

Note: you may need to restart the kernel to use updated packages.


In [3]:
dbutils.library.restartPython()

NameError: name 'dbutils' is not defined

Toy graph test

In [4]:
from graphframes import GraphFrame

v = spark.createDataFrame([("a",), ("b",), ("c",), ("d",), ("e",)], ["id"])
e = spark.createDataFrame([("a","b"), ("b","c"), ("c","a"), ("d","e")], ["src", "dst"])
g = GraphFrame(v, e)

print("inDegrees:")
g.inDegrees.show()

print("labelPropagation:")
g.labelPropagation(maxIter=5).show()

NameError: name 'spark' is not defined

Test every GraphFrames function

In [6]:
tests = {
    "outDegrees":          lambda: g.outDegrees,
    "degrees":             lambda: g.degrees,
    "triplets":            lambda: g.triplets,
    "find (motif)":        lambda: g.find("(x)-[]->(y); (y)-[]->(z)"),
    "filterEdges":         lambda: g.filterEdges("src = 'a'"),
    "bfs":                 lambda: g.bfs("id = 'a'", "id = 'c'"),
    "connectedComponents": lambda: g.connectedComponents(),
    "pageRank":            lambda: g.pageRank(resetProbability=0.15, maxIter=5).vertices,
    "shortestPaths":       lambda: g.shortestPaths(landmarks=["a"]),
    "triangleCount":       lambda: g.triangleCount(),
    "stronglyConnected":   lambda: g.stronglyConnectedComponents(maxIter=5),
}

results = []
for name, fn in tests.items():
    try:
        fn().limit(1).collect()
        results.append((name, "OK", ""))
    except Exception as ex:
        results.append((name, "FAIL", str(ex).split("\n")[0][:120]))

for name, status, err in results:
    print(f"{status:5} {name:20} {err}")

FAIL  outDegrees           name 'g' is not defined
FAIL  degrees              name 'g' is not defined
FAIL  triplets             name 'g' is not defined
FAIL  find (motif)         name 'g' is not defined
FAIL  filterEdges          name 'g' is not defined
FAIL  bfs                  name 'g' is not defined
FAIL  connectedComponents  name 'g' is not defined
FAIL  pageRank             name 'g' is not defined
FAIL  shortestPaths        name 'g' is not defined
FAIL  triangleCount        name 'g' is not defined
FAIL  stronglyConnected    name 'g' is not defined
